# Florida Hurricane Loss Analysis
## Wind/Water Attribution and Climate Scaling Visualization

This notebook provides interactive visualizations for:
1. Total loss overview with fixed wind/flood splits
2. Fixed vs county-specific wind/water attribution comparison
3. Wind and water loss components
4. Present and future climate scaling factors

**Event:** Great Miami Hurricane (1926)  
**Event ID:** 1926255N15314

## 1. Import Required Libraries

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import matplotlib.patches as mpatches

# Add parent directory to path to import fl_risk_model
sys.path.insert(0, str(Path().resolve().parent))

from fl_risk_model.loss_calc_utils.evaluate_losses import plot_event_county_map, county_totals_gdf
from fl_risk_model import config as cfg

# Set plotting style
plt.style.use('default')
%matplotlib inline

## 2. Load Data and Setup

Load exposure data with county information, Florida counties geodataframe, and empirical scaling factors.

In [ ]:
# Define paths
DATA_DIR = Path("../fl_risk_model/data")

# Event configuration
EVENT_ID = "1926255N15314"  # Great Miami Hurricane 1926
BASELINE_WIND_SHARE = 0.70   # Fixed 70% wind baseline

print("Loading data...")
print(f"Event: Great Miami Hurricane (1926)")
print(f"Event ID: {EVENT_ID}")
print(f"Baseline wind share: {BASELINE_WIND_SHARE:.0%}")

In [ ]:
# Load Florida counties geodataframe
county_fp = DATA_DIR / "US_counties"
counties = gpd.read_file(county_fp)
fl_counties = counties[counties["STATEFP"] == "12"].copy()

print(f"Loaded {len(fl_counties)} Florida counties")

# Load historical event county-level losses
event_file = DATA_DIR / "hazard" / "historical_events" / f"{EVENT_ID}.csv"
event_losses = pd.read_csv(event_file)
# Convert countyfp from float to zero-padded string to match shapefile format
# Drop any rows with NaN countyfp, then convert float -> int -> string with leading zeros
event_losses = event_losses.dropna(subset=['countyfp'])
event_losses['countyfp'] = event_losses['countyfp'].astype(int).astype(str).str.zfill(3)

print(f"Loaded event losses: {len(event_losses)} counties")
print(f"Total loss: ${event_losses['value'].sum()/1e9:.1f}B")

In [ ]:
# Load empirical wind/water attribution (present-day, 95th percentile)
attribution = pd.read_csv(DATA_DIR / "florida_empirical_hazard_attribution_p95.csv")
attribution['countyfp'] = attribution['admin2_name'].map(
    fl_counties.set_index('NAME')['COUNTYFP'].to_dict()
)
attribution['countyfp'] = attribution['countyfp'].astype(str).str.zfill(3)

# Load future scaling factors
future_scaling = pd.read_csv(DATA_DIR / "florida_future_damage_scaling_factors.csv")
future_scaling['countyfp'] = future_scaling['county_name'].map(
    fl_counties.set_index('NAME')['COUNTYFP'].to_dict()
)
future_scaling['countyfp'] = future_scaling['countyfp'].astype(str).str.zfill(3)

print(f"\nLoaded empirical attribution for {len(attribution)} counties")
print(f"Mean wind share (present): {attribution['wind_share'].mean():.1f}%")
print(f"\nLoaded future scaling factors for {len(future_scaling)} counties")
print(f"Mean damage scaling: {future_scaling['damage_scaling_total'].mean():.2f}x")

## 3. Define Helper Functions for Loss Calculations

In [ ]:
def create_loss_geodataframe(event_losses, fl_counties, loss_column='value', join_on='countyfp'):
    """Create a GeoDataFrame by merging event losses with county geometries."""
    gdf = fl_counties[['COUNTYFP', 'NAME', 'geometry']].copy()
    gdf['countyfp'] = gdf['COUNTYFP'].astype(str).str.zfill(3)
    gdf = gdf.rename(columns={'NAME': 'county_name'})
    
    # Merge with losses
    gdf = gdf.merge(event_losses[[join_on, loss_column]], on=join_on, how='left')
    return gdf


def apply_deviation_adjustment(event_losses, attribution, baseline_wind_share=0.70):
    """
    Apply deviation-based wind/water attribution.
    Formula: county_wind_share = baseline + (county_empirical - mean_empirical)
    """
    merged = event_losses.merge(attribution[['countyfp', 'wind_share']], on='countyfp', how='left')
    
    # Calculate mean wind share
    mean_wind = attribution['wind_share'].mean() / 100  # Convert to decimal
    
    # Apply deviation formula
    merged['wind_share_adjusted'] = baseline_wind_share + (merged['wind_share']/100 - mean_wind)
    merged['wind_loss'] = merged['value'] * merged['wind_share_adjusted']
    merged['water_loss'] = merged['value'] * (1 - merged['wind_share_adjusted'])
    
    return merged


def plot_choropleth(gdf, value_col, title, ax, cmap='cividis', logscale=True,
                   edgecolor='white', linewidth=0.6, missing_color='#f0f0f0'):
    """Plot a choropleth map on the given axes."""
    # Background
    gdf.plot(ax=ax, color=missing_color, linewidth=0, zorder=0)
    
    # Get values
    values = gdf[value_col].to_numpy(copy=False)
    
    # Choose normalization
    if logscale:
        positive = values[np.isfinite(values) & (values > 0)]
        vmin = positive.min() if positive.size else 1
        vmax = positive.max() if positive.size else 1
        norm = LogNorm(vmin=vmin, vmax=vmax)
    else:
        finite = values[np.isfinite(values)]
        vmin = np.nanmin(finite) if finite.size else 0
        vmax = np.nanmax(finite) if finite.size else 1
        norm = Normalize(vmin=vmin, vmax=vmax)
    
    # Plot filled polygons
    gdf.dropna(subset=[value_col]).plot(
        ax=ax, column=value_col, cmap=cmap, norm=norm,
        edgecolor=edgecolor, linewidth=linewidth, zorder=1
    )
    
    # County outlines
    gdf.boundary.plot(ax=ax, color=edgecolor, linewidth=linewidth, zorder=2)
    
    ax.set_axis_off()
    ax.set_title(title, pad=8, fontsize=10)
    
    return norm

print("Helper functions defined successfully!")

## 4. Prepare Loss Data

Calculate fixed and county-specific wind/water losses using the deviation approach.

In [ ]:
# Calculate fixed wind/flood losses (70/30 split)
event_losses['wind_loss_fixed'] = event_losses['value'] * BASELINE_WIND_SHARE
event_losses['flood_loss_fixed'] = event_losses['value'] * (1 - BASELINE_WIND_SHARE)

# Apply deviation-based county-specific adjustment
losses_with_attribution = apply_deviation_adjustment(event_losses, attribution, BASELINE_WIND_SHARE)

# Calculate differences
losses_with_attribution['wind_diff'] = losses_with_attribution['wind_loss'] - losses_with_attribution['value'] * BASELINE_WIND_SHARE
losses_with_attribution['flood_diff'] = losses_with_attribution['water_loss'] - losses_with_attribution['value'] * (1 - BASELINE_WIND_SHARE)

print("Loss calculations complete!")
print(f"\nTotal loss: ${event_losses['value'].sum()/1e9:.1f}B")
print(f"Fixed 70/30 split:")
print(f"  Wind (70%): ${event_losses['wind_loss_fixed'].sum()/1e9:.1f}B")
print(f"  Flood (30%): ${event_losses['flood_loss_fixed'].sum()/1e9:.1f}B")
print(f"\nCounty-specific (deviation approach):")
print(f"  Wind: ${losses_with_attribution['wind_loss'].sum()/1e9:.1f}B ({losses_with_attribution['wind_loss'].sum()/losses_with_attribution['value'].sum()*100:.1f}%)")
print(f"  Water: ${losses_with_attribution['water_loss'].sum()/1e9:.1f}B ({losses_with_attribution['water_loss'].sum()/losses_with_attribution['value'].sum()*100:.1f}%)")

In [ ]:
# Verify Beta parameters for all scenarios
import pandas as pd

# Load empirical data
df_emp = pd.read_csv('../../fl_risk_model/data/florida_future_scaling_factors_p95.csv')
multiplicative_ratio = df_emp['wind_share_future'].mean() / df_emp['wind_share_present'].mean()

print('='*80)
print('BETA PARAMETERS VERIFICATION')
print('='*80)
print(f'\nEmpirical multiplicative ratio: {multiplicative_ratio:.4f}')
print(f'Interpretation: Wind share decreases to {multiplicative_ratio*100:.1f}% of present value')
print(f'               (Equivalent to {(1-multiplicative_ratio)*100:.1f}pp shift toward water)\n')

# Define scenarios
scenarios = [
    ('Great Miami (1926)', '1926255N15314', 0.70, 10),
    ('Andrew (1992)', '1992230N11325', 0.875, 10),
    ('Lake Okeechobee (1928)', '1928250N14343', 0.30, 8),
    ('Irma (2017)', '2017242N16333', 0.50, 10),
]

print('SINGLE EVENT SCENARIOS')
print('-' * 80)
for name, event_id, mean_pres, conc in scenarios:
    mean_fut = mean_pres * multiplicative_ratio
    alpha_pres, beta_pres = mean_pres * conc, (1 - mean_pres) * conc
    alpha_fut, beta_fut = mean_fut * conc, (1 - mean_fut) * conc
    
    print(f'\n{name}:')
    print(f'  Event ID: {event_id}')
    print(f'  Present:  Beta(α={alpha_pres:.2f}, β={beta_pres:.2f}) → {mean_pres*100:.1f}% wind')
    print(f'  Future:   Beta(α={alpha_fut:.2f}, β={beta_fut:.2f}) → {mean_fut*100:.1f}% wind')
    print(f'  Change:   {(mean_fut-mean_pres)*100:+.1f}pp')

# Composite scenarios
composites = [
    ('Andrew → Great Miami', 'andrew_then_gm', 0.77, 25),
    ('Great Miami → Andrew', 'gm_then_andrew', 0.76, 25),
    ('Double Great Miami', 'double_gm', 0.70, 30),
    ('Double Irma', 'double_irma', 0.50, 30),
]

print('\n\nCOMPOSITE SCENARIOS')
print('-' * 80)
for name, scenario_id, mean_pres, conc in composites:
    mean_fut = mean_pres * multiplicative_ratio
    alpha_pres, beta_pres = mean_pres * conc, (1 - mean_pres) * conc
    alpha_fut, beta_fut = mean_fut * conc, (1 - mean_fut) * conc
    
    print(f'\n{name}:')
    print(f'  Scenario ID: {scenario_id}')
    print(f'  Present:  Beta(α={alpha_pres:.2f}, β={beta_pres:.2f}) → {mean_pres*100:.1f}% wind')
    print(f'  Future:   Beta(α={alpha_fut:.2f}, β={beta_fut:.2f}) → {mean_fut*100:.1f}% wind')
    print(f'  Change:   {(mean_fut-mean_pres)*100:+.1f}pp')

print('\n' + '='*80)
print('These parameters are now implemented in:')
print('  - fl_risk_model/config.py')
print('  - fl_risk_model/mc_run_events.py')
print('='*80)

In [ ]:
# Calculate future losses by applying scaling factors to present-day county-specific losses
gdf_future_losses = gdf_attr[['countyfp', 'county_name', 'geometry', 'wind_loss', 'water_loss']].copy()

# Merge with future scaling factors
gdf_future_losses = gdf_future_losses.merge(
    future_scaling[['countyfp', 'damage_scaling_wind', 'damage_scaling_rain', 'damage_scaling_surge']], 
    on='countyfp', how='left'
)

# Merge with attribution data to get rain/surge shares for weighted averaging
gdf_future_losses = gdf_future_losses.merge(
    attribution[['countyfp', 'rain_share', 'surge_share']], 
    on='countyfp', how='left'
)

# Calculate weighted water scaling based on rain/surge composition
# water_scaling = (rain_share × rain_scaling + surge_share × surge_scaling) / (rain_share + surge_share)
water_share_total = gdf_future_losses['rain_share'] + gdf_future_losses['surge_share']
gdf_future_losses['damage_scaling_water'] = (
    gdf_future_losses['rain_share'] * gdf_future_losses['damage_scaling_rain'] +
    gdf_future_losses['surge_share'] * gdf_future_losses['damage_scaling_surge']
) / water_share_total

# Calculate future losses
gdf_future_losses['wind_loss_future'] = gdf_future_losses['wind_loss'] * gdf_future_losses['damage_scaling_wind']
gdf_future_losses['water_loss_future'] = gdf_future_losses['water_loss'] * gdf_future_losses['damage_scaling_water']
gdf_future_losses['total_loss_future'] = gdf_future_losses['wind_loss_future'] + gdf_future_losses['water_loss_future']

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))

# a) Total future loss
norm = plot_choropleth(gdf_future_losses, 'total_loss_future',
                'Total Loss (Future)\n(SSP2-4.5, 2070-2100)',
                axes[0], cmap='cividis', logscale=True)
axes[0].text(-0.01, 1.1, 'a', transform=axes[0].transAxes, 
             fontsize=12, fontweight='bold', va='top', ha='left')

# b) Wind component (future)
plot_choropleth(gdf_future_losses, 'wind_loss_future',
                'Wind Loss (Future)\n(SSP2-4.5, 2070-2100)',
                axes[1], cmap='cividis', logscale=True)
axes[1].text(-0.01, 1.1, 'b', transform=axes[1].transAxes, 
             fontsize=12, fontweight='bold', va='top', ha='left')

# c) Water component (future)
plot_choropleth(gdf_future_losses, 'water_loss_future',
                'Water Loss (Future)\n(SSP2-4.5, 2070-2100)',
                axes[2], cmap='cividis', logscale=True)
axes[2].text(-0.01, 1.1, 'c', transform=axes[2].transAxes, 
             fontsize=12, fontweight='bold', va='top', ha='left')

# Add colorbar on the right side (vertical)
sm = plt.cm.ScalarMappable(cmap='cividis', norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.92, 0.2, 0.015, 0.6])  # [left, bottom, width, height]
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Loss ($)', fontsize=10)

plt.subplots_adjust(wspace=0.05)
plt.show()

total_future = gdf_future_losses['total_loss_future'].sum()
wind_future = gdf_future_losses['wind_loss_future'].sum()
water_future = gdf_future_losses['water_loss_future'].sum()

print(f"Total: ${total_future/1e9:.1f}B (100%)")
print(f"Wind:  ${wind_future/1e9:.1f}B ({wind_future/total_future*100:.1f}%)")
print(f"Water: ${water_future/1e9:.1f}B ({water_future/total_future*100:.1f}%)")
print(f"\nScaling from present day:")
print(f"Total: {total_future/total:.2f}x")
print(f"Wind:  {wind_future/wind:.2f}x")
print(f"Water: {water_future/water:.2f}x")

## 9. PLOT 6: Future Losses and Scaling Factors (2×3 grid)

Future climate losses and relative scaling factors.

**Row 1:** Future losses (total, wind, water) under SSP2-4.5, 2070-2100  
**Row 2:** Future scaling factors (total, wind, water) - shows multiplicative increase from present-day baseline

In [ ]:
# Calculate changes (future - present)
gdf_future_losses['total_change'] = gdf_future_losses['total_loss_future'] - (gdf_attr['wind_loss'] + gdf_attr['water_loss'])
gdf_future_losses['wind_change'] = gdf_future_losses['wind_loss_future'] - gdf_attr['wind_loss']
gdf_future_losses['water_change'] = gdf_future_losses['water_loss_future'] - gdf_attr['water_loss']

# Calculate relative changes (scaling factors)
gdf_future_losses['total_scaling'] = gdf_future_losses['total_loss_future'] / (gdf_attr['wind_loss'] + gdf_attr['water_loss'])
gdf_future_losses['wind_scaling'] = gdf_future_losses['wind_loss_future'] / gdf_attr['wind_loss']
gdf_future_losses['water_scaling'] = gdf_future_losses['water_loss_future'] / gdf_attr['water_loss']

fig, axes = plt.subplots(2, 3, figsize=(10, 7))

# ROW 1: Future losses
# a) Future total
norm_present = plot_choropleth(gdf_future_losses, 'total_loss_future',
                'Future: Total Loss\n(SSP2-4.5, 2070-2100)',
                axes[0, 0], cmap='cividis', logscale=True)
axes[0, 0].text(-0.01, 1.1, 'a', transform=axes[0, 0].transAxes, 
                fontsize=12, fontweight='bold', va='top', ha='left')

# b) Future wind
plot_choropleth(gdf_future_losses, 'wind_loss_future',
                'Future: Wind Loss\n(SSP2-4.5, 2070-2100)',
                axes[0, 1], cmap='cividis', logscale=True)
axes[0, 1].text(-0.01, 1.1, 'b', transform=axes[0, 1].transAxes, 
                fontsize=12, fontweight='bold', va='top', ha='left')

# c) Future water
plot_choropleth(gdf_future_losses, 'water_loss_future',
                'Future: Water Loss\n(SSP2-4.5, 2070-2100)',
                axes[0, 2], cmap='cividis', logscale=True)
axes[0, 2].text(-0.01, 1.1, 'c', transform=axes[0, 2].transAxes, 
                fontsize=12, fontweight='bold', va='top', ha='left')

# ROW 2: Future change (absolute increase)
# d) Total change
norm_change = plot_choropleth(gdf_future_losses, 'total_scaling',
                'Future Scaling: Total\n(SSP2-4.5, 2070-2100)',
                axes[1, 0], cmap='Reds', logscale=False)
axes[1, 0].text(-0.01, 1.1, 'd', transform=axes[1, 0].transAxes, 
                fontsize=12, fontweight='bold', va='top', ha='left')

# e) Wind change
plot_choropleth(gdf_future_losses, 'wind_scaling',
                'Future Scaling: Wind\n(SSP2-4.5, 2070-2100)',
                axes[1, 1], cmap='Reds', logscale=False)
axes[1, 1].text(-0.01, 1.1, 'e', transform=axes[1, 1].transAxes, 
                fontsize=12, fontweight='bold', va='top', ha='left')

# f) Water change
plot_choropleth(gdf_future_losses, 'water_scaling',
                'Future Scaling: Water\n(SSP2-4.5, 2070-2100)',
                axes[1, 2], cmap='Reds', logscale=False)
axes[1, 2].text(-0.01, 1.1, 'f', transform=axes[1, 2].transAxes, 
                fontsize=12, fontweight='bold', va='top', ha='left')

# Add colorbar for row 1 (present-day losses - vertical, on right side)
sm_present = plt.cm.ScalarMappable(cmap='cividis', norm=norm_present)
sm_present.set_array([])
cbar_ax1 = fig.add_axes([0.92, 0.535, 0.015, 0.35])  # [left, bottom, width, height]
cbar1 = fig.colorbar(sm_present, cax=cbar_ax1)
cbar1.set_label('Loss (USD)', fontsize=10)

# Add colorbar for row 2 (future change - vertical, on right side)
sm_change = plt.cm.ScalarMappable(cmap='Reds', norm=norm_change)
sm_change.set_array([])
cbar_ax2 = fig.add_axes([0.92, 0.1, 0.015, 0.35])  # [left, bottom, width, height]
cbar2 = fig.colorbar(sm_change, cax=cbar_ax2)
cbar2.set_label('Scaling Factor (×)', fontsize=10)

plt.subplots_adjust(wspace=0.05)
plt.show()

print("Future losses (SSP2-4.5, 2070-2100):")
print(f"  Total: ${total_future/1e9:.1f}B")
print(f"  Wind:  ${wind_future/1e9:.1f}B ({wind_future/total_future*100:.1f}%)")
print(f"  Water: ${water_future/1e9:.1f}B ({water_future/total_future*100:.1f}%)")
print("\nFuture scaling factors (relative to present-day baseline):")
print(f"  Total: {gdf_future_losses['total_scaling'].mean():.3f}x (range: {gdf_future_losses['total_scaling'].min():.3f}x-{gdf_future_losses['total_scaling'].max():.3f}x)")
print(f"  Wind:  {gdf_future_losses['wind_scaling'].mean():.3f}x (range: {gdf_future_losses['wind_scaling'].min():.3f}x-{gdf_future_losses['wind_scaling'].max():.3f}x)")
print(f"  Water: {gdf_future_losses['water_scaling'].mean():.3f}x (range: {gdf_future_losses['water_scaling'].min():.3f}x-{gdf_future_losses['water_scaling'].max():.3f}x)")
print("\nAbsolute change from present day:")
print(f"  Total: ${gdf_future_losses['total_change'].sum()/1e9:.1f}B ({gdf_future_losses['total_change'].sum()/total*100:.1f}% increase)")
print(f"  Wind:  ${gdf_future_losses['wind_change'].sum()/1e9:.1f}B ({gdf_future_losses['wind_change'].sum()/wind*100:.1f}% increase)")
print(f"  Water: ${gdf_future_losses['water_change'].sum()/1e9:.1f}B ({gdf_future_losses['water_change'].sum()/water*100:.1f}% increase)")

In [ ]:
# Fine-tuned future loss and scaling plot (no figure titles, annotated with values and scaling)
# Automatically save figure to results/figures

import os

figures_dir = os.path.abspath(os.path.join('..', '..', 'results', 'figures'))

os.makedirs(figures_dir, exist_ok=True)

# Calculate changes (future - present)
gdf_future_losses['total_change'] = gdf_future_losses['total_loss_future'] - (gdf_attr['wind_loss'] + gdf_attr['water_loss'])
gdf_future_losses['wind_change'] = gdf_future_losses['wind_loss_future'] - gdf_attr['wind_loss']
gdf_future_losses['water_change'] = gdf_future_losses['water_loss_future'] - gdf_attr['water_loss']

# Calculate relative changes (scaling factors)
gdf_future_losses['total_scaling'] = gdf_future_losses['total_loss_future'] / (gdf_attr['wind_loss'] + gdf_attr['water_loss'])
gdf_future_losses['wind_scaling'] = gdf_future_losses['wind_loss_future'] / gdf_attr['wind_loss']
gdf_future_losses['water_scaling'] = gdf_future_losses['water_loss_future'] / gdf_attr['water_loss']

fig, axes = plt.subplots(2, 3, figsize=(10, 7))

# ROW 1: Future losses (no titles)
# a) Total
norm_present = plot_choropleth(gdf_future_losses, 'total_loss_future', '', axes[0, 0], cmap='cividis', logscale=True)
axes[0, 0].text(-0.05, 1, 'a', transform=axes[0, 0].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')
# b) Wind
plot_choropleth(gdf_future_losses, 'wind_loss_future', '', axes[0, 1], cmap='cividis', logscale=True)
axes[0, 1].text(-0.05, 1, 'b', transform=axes[0, 1].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')
# c) Water
plot_choropleth(gdf_future_losses, 'water_loss_future', '', axes[0, 2], cmap='cividis', logscale=True)
axes[0, 2].text(-0.05, 1, 'c', transform=axes[0, 2].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

# ROW 2: Future scaling (no titles)
# d) Total scaling
norm_change = plot_choropleth(gdf_future_losses, 'total_scaling', '', axes[1, 0], cmap='Reds', logscale=False)
axes[1, 0].text(-0.05, 1, 'd', transform=axes[1, 0].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')
# e) Wind scaling
plot_choropleth(gdf_future_losses, 'wind_scaling', '', axes[1, 1], cmap='Reds', logscale=False)
axes[1, 1].text(-0.05, 1, 'e', transform=axes[1, 1].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')
# f) Water scaling
plot_choropleth(gdf_future_losses, 'water_scaling', '', axes[1, 2], cmap='Reds', logscale=False)
axes[1, 2].text(-0.05, 1, 'f', transform=axes[1, 2].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

# --- Annotate values and scaling factors ---
# Precompute values for annotation
future_vals = [
    gdf_future_losses['total_loss_future'].sum(),
    gdf_future_losses['wind_loss_future'].sum(),
    gdf_future_losses['water_loss_future'].sum()
    ]
present_vals = [
    (gdf_attr['wind_loss'] + gdf_attr['water_loss']).sum(),
    gdf_attr['wind_loss'].sum(),
    gdf_attr['water_loss'].sum()
    ]
changes = [
    gdf_future_losses['total_change'].sum(),
    gdf_future_losses['wind_change'].sum(),
    gdf_future_losses['water_change'].sum()
    ]
change_perc = [
    changes[0] / present_vals[0] * 100 if present_vals[0] else 0,
    changes[1] / present_vals[1] * 100 if present_vals[1] else 0,
    changes[2] / present_vals[2] * 100 if present_vals[2] else 0
    ]

scalings = [
    gdf_future_losses['total_scaling'].mean(),
    gdf_future_losses['wind_scaling'].mean(),
    gdf_future_losses['water_scaling'].mean()
    ]
scaling_ranges = [
    (gdf_future_losses['total_scaling'].min(), gdf_future_losses['total_scaling'].max()),
    (gdf_future_losses['wind_scaling'].min(), gdf_future_losses['wind_scaling'].max()),
    (gdf_future_losses['water_scaling'].min(), gdf_future_losses['water_scaling'].max())
    ]

# Panel a, b, c: annotate with future value, %, and change from present
tot = future_vals[0]
wind = future_vals[1]
water = future_vals[2]
for i, ax in enumerate(axes[0]):
    val = future_vals[i]
    perc = val / tot * 100 if tot else 0
    abs_change = changes[i]
    abs_perc = change_perc[i]
    ax.annotate(
        f"State-wide total:\n ${val/1e9:.1f}B ({perc:.1f}%)",
        xy=(0.4, 0.2), xycoords='axes fraction', ha='center', va='top', fontsize=8
    )

# Panel d, e, f: annotate with scaling factor and range
for i, ax in enumerate(axes[1]):
    mean = scalings[i]
    rng = scaling_ranges[i]
    ax.annotate(
        f"Mean future scaling factor:\n {mean:.2f}x ({rng[0]:.2f}–{rng[1]:.2f}x)",
        xy=(0.4, 0.2), xycoords='axes fraction', ha='center', va='top', fontsize=8
    )

# Add colorbar for row 1 (present-day losses - vertical, on right side)
sm_present = plt.cm.ScalarMappable(cmap='cividis', norm=norm_present)
sm_present.set_array([])
cbar_ax1 = fig.add_axes([0.92, 0.585, 0.015, 0.25])  # [left, bottom, width, height]
cbar1 = fig.colorbar(sm_present, cax=cbar_ax1)
cbar1.set_label('Loss (USD)', fontsize=10)

# Add colorbar for row 2 (future change - vertical, on right side)
sm_change = plt.cm.ScalarMappable(cmap='Reds', norm=norm_change)
sm_change.set_array([])
cbar_ax2 = fig.add_axes([0.92, 0.17, 0.015, 0.25])  # [left, bottom, width, height]
cbar2 = fig.colorbar(sm_change, cax=cbar_ax2)
cbar2.set_label('Scaling Factor (×)', fontsize=10)

plt.subplots_adjust(wspace=0.05)

# Save the figure

fig_path = os.path.join(figures_dir, 'figS2_future_loss_scaling_gm.png')

fig.savefig(fig_path, dpi=300, bbox_inches='tight')

print(f'Figure saved to: {fig_path}')

plt.show()

In [ ]:
# Plot: Scaling factors (no figure titles, direct save to outputs)
import os
figures_dir = os.path.abspath(os.path.join('..', '..', 'results', 'figures'))
os.makedirs(figures_dir, exist_ok=True)

# Prepare geodataframes for scaling factors
gdf_present = fl_counties[['COUNTYFP', 'NAME', 'geometry']].copy()
gdf_present['countyfp'] = gdf_present['COUNTYFP'].astype(str).str.zfill(3)

# Calculate deviations from empirical mean (centered at 0)
empirical_mean_wind = attribution['wind_share'].mean()
attribution_with_dev = attribution.copy()
attribution_with_dev['wind_dev_from_mean'] = attribution_with_dev['wind_share'] - empirical_mean_wind
attribution_with_dev['water_dev_from_mean'] = -attribution_with_dev['wind_dev_from_mean']

gdf_present = gdf_present.merge(attribution_with_dev[['countyfp', 'wind_dev_from_mean', 'water_dev_from_mean', 'rain_share', 'surge_share']], 
                                 on='countyfp', how='left')

# Calculate weighted water scaling for future based on rain/surge composition
future_scaling_with_water = future_scaling.merge(
    attribution[['countyfp', 'rain_share', 'surge_share']], 
    on='countyfp', how='left'
)
water_share_total = future_scaling_with_water['rain_share'] + future_scaling_with_water['surge_share']
future_scaling_with_water['damage_scaling_water'] = (
    future_scaling_with_water['rain_share'] * future_scaling_with_water['damage_scaling_rain'] +
    future_scaling_with_water['surge_share'] * future_scaling_with_water['damage_scaling_surge']
) / water_share_total

gdf_future = fl_counties[['COUNTYFP', 'NAME', 'geometry']].copy()
gdf_future['countyfp'] = gdf_future['COUNTYFP'].astype(str).str.zfill(3)
gdf_future = gdf_future.merge(future_scaling_with_water[['countyfp', 'damage_scaling_wind', 'damage_scaling_water']], 
                               on='countyfp', how='left')

fig, axes = plt.subplots(2, 2, figsize=(7, 7))

# ROW 1: Present-day scaling factors (relative to 70/30 baseline) - use RdBu_r for both
# a) Present wind scaling
norm1 = plot_choropleth(gdf_present, 'wind_dev_from_mean', '', axes[0, 0], cmap='RdBu_r', logscale=False)
axes[0, 0].text(-0.02, 1, 'a', transform=axes[0, 0].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

# b) Present water scaling
plot_choropleth(gdf_present, 'water_dev_from_mean', '', axes[0, 1], cmap='RdBu_r', logscale=False)
axes[0, 1].text(-0.02, 1, 'b', transform=axes[0, 1].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

# Add colorbar for row 1 (vertical, on right side)
sm1 = plt.cm.ScalarMappable(cmap='RdBu_r', norm=norm1)
sm1.set_array([])
cbar_ax1 = fig.add_axes([0.98, 0.59, 0.02, 0.3])  # [left, bottom, width, height]
cbar1 = fig.colorbar(sm1, cax=cbar_ax1)
cbar1.set_label('Deviation (percentage points)', fontsize=10)

# ROW 2: Future climate scaling factors (SSP2-4.5, 2070-2100) - use Reds for both
# c) Future wind scaling
norm2 = plot_choropleth(gdf_future, 'damage_scaling_wind', '', axes[1, 0], cmap='Reds', logscale=False)
axes[1, 0].text(-0.02, 1, 'c', transform=axes[1, 0].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

# d) Future water (rain+surge) scaling
plot_choropleth(gdf_future, 'damage_scaling_water', '', axes[1, 1], cmap='Reds', logscale=False)
axes[1, 1].text(-0.02, 1, 'd', transform=axes[1, 1].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

# Add colorbar for row 2 (vertical, on right side)
sm2 = plt.cm.ScalarMappable(cmap='Reds', norm=norm2)
sm2.set_array([])
cbar_ax2 = fig.add_axes([0.98, 0.12, 0.02, 0.3])  # [left, bottom, width, height]
cbar2 = fig.colorbar(sm2, cax=cbar_ax2)
cbar2.set_label('Damage Scaling Factor (×)', fontsize=10)

# --- Annotate with summary values ---
# Compute values
wind_mean = gdf_future['damage_scaling_wind'].mean()
wind_min = gdf_future['damage_scaling_wind'].min()
wind_max = gdf_future['damage_scaling_wind'].max()
water_mean = gdf_future['damage_scaling_water'].mean()
water_min = gdf_future['damage_scaling_water'].min()
water_max = gdf_future['damage_scaling_water'].max()

# Add range to a and b
axes[0, 0].annotate(f"Range: {gdf_present['wind_dev_from_mean'].min():.3f}%-{gdf_present['wind_dev_from_mean'].max():.3f}%", xy=(0.4, 0.2), xycoords='axes fraction', ha='center', va='bottom', fontsize=9)
axes[0, 1].annotate(f"Range: {gdf_present['water_dev_from_mean'].min():.3f}%-{gdf_present['water_dev_from_mean'].max():.3f}%", xy=(0.4, 0.2), xycoords='axes fraction', ha='center', va='bottom', fontsize=9)

# Add mean and range to c and d
axes[1, 0].annotate(f"Wind loss scaling:\n{wind_mean:.3f}x (range: {wind_min:.3f}x–{wind_max:.3f}x)", xy=(0.4, 0.2), xycoords='axes fraction', ha='center', va='bottom', fontsize=9)
axes[1, 1].annotate(f"Flood (rain+surge) loss scaling:\n{water_mean:.3f}x (range: {water_min:.3f}x–{water_max:.3f}x)", xy=(0.4, 0.2), xycoords='axes fraction', ha='center', va='bottom', fontsize=9)

plt.tight_layout()

# Save the figure directly to outputs
fig_path = os.path.join(figures_dir, 'figS1_scaling_factors.png')
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f'Figure saved to: {fig_path}')
plt.show()
